In [194]:
import os
import sys 
sys.path.append(os.getcwd())
from pathlib import Path
import pandas as pd 
import numpy as np
# ==========================================
# 1. CORE DIRECTORIES
# ==========================================
ONEDRIVE_ROOT = Path(os.environ.get('OneDrive', ''))
DATA_ROOT = ONEDRIVE_ROOT / '0. DATASETS'
RAW_DATA_DIR = DATA_ROOT / 'raw'
OUTPUT_DIR = DATA_ROOT / 'outputs'
PROJECT_ROOT = OUTPUT_DIR / 'Direct-Investing-TLH'

# ==========================================
# 2. PIPELINE SUBDIRECTORIES
# ==========================================
# Creating the dedicated output folders for each stage of the new engine
DATA_DIR = PROJECT_ROOT / '01_Data'


In [195]:
# Loading our daily market data 
df_univ = pd.read_parquet(DATA_DIR / 'tlh_universe.parquet')
# choosing the period
start_date = pd.to_datetime('2015-01-01')
end_date = pd.to_datetime ('2023-12-31')
df_univ = df_univ[(df_univ['date'] >= start_date) & 
                  (df_univ['date'] <= end_date)]
df_univ.set_index('date', inplace=True)
df_univ

,permno,siccd,dlyret,dlyretx,sprtrn,currency,true_divisor_prc,true_divisor_shr,split_event_prc,split_event_shr,cfacpr,cfacshr,prc_adj_usd,shrout_adj,mkt_cap_usd,divamt_net_usd,usd_cad,prc_adj_cad,divamt_net_cad
date,,,,,,,,,,,,,,,,,,,
2015-01-02,10104,7372.0,-0.014232,-0.014232,-0.000340,USD,1.0,1.0,1.0,1.0,1.0,1.0,44.33,4391367.0,1.946693e+08,0.000,1.1725,51.976925,0.000000
2015-01-05,10104,7372.0,-0.013986,-0.016693,-0.018278,USD,1.0,1.0,1.0,1.0,1.0,1.0,43.59,4391367.0,1.914197e+08,0.102,1.1785,51.370815,0.120207
2015-01-06,10104,7372.0,-0.010323,-0.010323,-0.008893,USD,1.0,1.0,1.0,1.0,1.0,1.0,43.14,4391367.0,1.894436e+08,0.000,1.1802,50.913828,0.000000
2015-01-07,10104,7372.0,0.000232,0.000232,0.011630,USD,1.0,1.0,1.0,1.0,1.0,1.0,43.15,4391367.0,1.894875e+08,0.000,1.1851,51.137065,0.000000
2015-01-08,10104,7372.0,0.006025,0.006025,0.017888,USD,1.0,1.0,1.0,1.0,1.0,1.0,43.41,4391367.0,1.906292e+08,0.000,1.1812,51.275892,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-22,93436,3711.0,-0.007701,-0.007701,0.001660,USD,1.0,1.0,1.0,1.0,1.0,1.0,252.54,3178921.0,8.028047e+08,0.000,1.3259,334.842786,0.000000
2023-12-26,93436,3711.0,0.016116,0.016116,0.004232,USD,1.0,1.0,1.0,1.0,1.0,1.0,256.61,3178921.0,8.157429e+08,0.000,1.3208,338.930488,0.000000
2023-12-27,93436,3711.0,0.018822,0.018822,0.001430,USD,1.0,1.0,1.0,1.0,1.0,1.0,261.44,3178921.0,8.310971e+08,0.000,1.3201,345.126944,0.000000


In [196]:
trading_days = df_univ.index.unique().sort_values()
# first day of trading
day_1 = trading_days[0]
# market data on day 1
daily_data_1 = df_univ.loc[day_1]

benchmark_permnos = daily_data_1['permno'].values

# dictionaries are much faster than .loc on series or dfs. 
# creating dictionaries for prices and dividends 
prices_cad = dict(zip(daily_data_1['permno'], daily_data_1['prc_adj_cad']))
divs_cad = dict(zip(daily_data_1['permno'], daily_data_1['divamt_net_cad']))

total_mkt_cap = daily_data_1['mkt_cap_usd'].sum()

# benchmark weights 
weights = daily_data_1['mkt_cap_usd'] / total_mkt_cap
bench_w = pd.Series(weights.values, index = daily_data_1['permno'])

In [197]:
bench_w

permno
10104    0.011836
10107    0.023364
10138    0.001366
10145    0.004553
10147         NaN
           ...   
92988         NaN
93002    0.001554
93096    0.001306
93159    0.000430
93422         NaN
Length: 502, dtype: float64

In [198]:


# CRA 30-Day Superficial Loss Trackers
# when we sell a stock, we are going to record the date of the sale so that we don't by back the stock
superficial_loss_lockouts = {}  # Forward rule: {permno: expiration_date}

initial_capital_cad = 1_000_000
# recording cash 
# initial cash (cash at time 0 ) is initial capital 
cash_cad = initial_capital_cad

# DAY 1: INITIALIZATION (Buying the Benchmark)
print(f"--- Processing Day 1: {day_1.date()} ---")

# 1. We want to perfectly track the benchmark on Day 1.
# Target dollars = benchmark weight * initial cash
# target_shares = {permno: shares}

target_shares = {}
for permno, weight in bench_w.items():
    # failsafe
    if pd.isna(weight) or weight <= 0:
        continue
    # get current price 
    price = prices_cad[permno]
    
    # failsafe: Skip if price is missing, NaN, or zero
    if price is None or pd.isna(price) or price <= 0:
        
        continue

    target_dollars = weight * initial_capital_cad
    # Truncate to 4 decimals for Wealthsimple fractional shares
    raw_shares = target_dollars / price
    target_shares[permno] = np.floor(raw_shares * 10000.0) / 10000.0

--- Processing Day 1: 2015-01-02 ---


In [199]:
dict(list(target_shares.items())[:5])

{10104: 227.7116,
 10107: 426.1393,
 10138: 13.5896,
 10145: 40.5921,
 10516: 33.3832}

In [200]:
# EXECUTING BUYS ON DAY 1
trade_history =[]               # Audit Trail
last_buy_dates = {}             # Recording for Backward rule: {permno: last_purchase_date}
do_not_harvest = []             # a list of permnos not to harvest because they were bought within the last 30 days
positions = {}                  # inititate our portfolio (empty on day zero)

# mark-to-Market : initiate value of the portfolio (0 on day 0)
total_equity_cad_1 = 0.0

# 2. Execute the "Buys" to establish the ACB and deplete Cash
for permno, shares in target_shares.items():
    # failsafe 
    if pd.isna(shares) or shares <= 0:
        continue
    price = prices_cad[permno]
    
    #failsafe
    if price is None or pd.isna(price) or price <= 0:
        continue

    cost = shares * price
    # add to aum
    total_equity_cad_1 += cost 
    # Deduct cash
    cash_cad -= cost
    # Record the position and the ACB
    positions[permno] = {
        'shares': shares, 
        'acb_per_share': price  # The ACB is locked in on Day 1
    }
    
    # Log the 30-Day Backward Rule (Cannot harvest if bought in last 30 days)
    last_buy_dates[permno] = day_1
total_aum_1 = total_equity_cad_1 + cash_cad

do_not_harvest =[
    permno for permno, last_buy in last_buy_dates.items()
    if (day_1 - last_buy).days <= 30
]

print(f"Number of positions established: {len(positions)}")
print(f"Total AUM @ the End of Day 1:  ${total_aum_1}")
print(f"End of Day 1 Cash: ${cash_cad:,.2f}")
print(f"Number of stocks we are not allowed to harvest @ the end of Day 1 : {len(do_not_harvest)}")


Number of positions established: 410
Total AUM @ the End of Day 1:  $1000000.0000000023
End of Day 1 Cash: $2.05
Number of stocks we are not allowed to harvest @ the end of Day 1 : 410


In [201]:
# sample positions 
dict(list(positions.items())[:5])

{10104: {'shares': 227.7116, 'acb_per_share': 51.976925},
 10107: {'shares': 426.1393, 'acb_per_share': 54.826100000000004},
 10138: {'shares': 13.5896, 'acb_per_share': 100.55360000000002},
 10145: {'shares': 40.5921, 'acb_per_share': 112.16021300555786},
 10516: {'shares': 33.3832, 'acb_per_share': 60.99345000000001}}

In [202]:
# DAY 2: THE MARKET MOVES (Scanning for Losses)
# Advance the clock to the next trading day
# Calculate AUM before possible rebalancing  
day_2 = trading_days[1]
print(f"\n--- Processing Day 2: {day_2.date()} ---")

# 1. Get the new state of the world
daily_data_2 = df_univ.loc[day_2].copy()
daily_data_2 = daily_data_2.replace([np.inf, -np.inf], np.nan)

prices_cad_2 = dict(zip(daily_data_2['permno'], daily_data_2['prc_adj_cad']))

# 2. Mark-to-Market (Calculate new Total AUM based on Day 2 prices)
total_equity_cad_2 = 0.0

# going over every position from the previous period 
for permno, pos in positions.items():

    shares = pos['shares']
    
    if pd.isna(shares) or shares is None: 
        continue 
    # Failsafe: if price is missing, use ACB 
    price_today = prices_cad_2.get(permno, pos['acb_per_share']) 
    total_equity_cad_2 += shares * price_today



--- Processing Day 2: 2015-01-05 ---


In [203]:
#========================
# DIVIDENDS 
divs_cad_2 = dict(zip(daily_data_2['permno'], daily_data_2['divamt_net_cad']))
# initialize day 40 dividends 
daily_dividend_income = 0.0
# Loop through what we own and see if they paid us today
for permno, pos in positions.items():
    dps = divs_cad_2.get(permno, 0.0)
    if dps > 0:
        payout = pos['shares'] * dps
        cash_cad += payout  # Add to our global cash balance!
        daily_dividend_income += payout

if daily_dividend_income > 0:
    print(f"Collected ${daily_dividend_income:,.2f} in CAD dividends on {day_2.date()}.")

Collected $57.12 in CAD dividends on 2015-01-05.


In [204]:
total_aum_2 = total_equity_cad_2 + cash_cad
portfolio_return_2 = (total_aum_2 - total_aum_1) / total_aum_1
print(f"Day 2 Total AUM: ${total_aum_2:,.2f}")
print(f"Portfolio Return @ Day 2: ${portfolio_return_2:,.4f}")


Day 2 Total AUM: $987,033.24
Portfolio Return @ Day 2: $-0.0130


In [205]:
# 3. THE HARVEST SCANNER
# We loop through our positions and check if anything dropped below our threshold (-5%) 
# and collect them in harvest_candidates  

# First we update our do not harvest list 
do_not_harvest =[
    permno for permno, last_buy in last_buy_dates.items()
    if (day_2 - last_buy).days <= 30
]

harvest_threshold = -0.05
harvest_candidates =[]
dropped_more_than_threshold = 0

for permno, pos in positions.items():
    acb = pos['acb_per_share']
    price_today = prices_cad_2.get(permno, acb)
    
    return_pct = (price_today - acb) / acb
    
    if return_pct <= harvest_threshold:
        dropped_more_than_threshold += 1 
        #check if permno is allowed to be harvested 
        if permno not in do_not_harvest:
            harvest_candidates.append(permno)

print(f"Number of stocks down more than 5% today: {dropped_more_than_threshold}")
print(f"Number of stocks eligibale for Tax Loss Harvesting: {len(harvest_candidates)}")


Number of stocks down more than 5% today: 18
Number of stocks eligibale for Tax Loss Harvesting: 0


In [206]:
# FAST FORWARD: DAY 40 (lockouts are expired)
# ==========================================
# We jump ahead ~40 trading days so the CRA 30-day backward rule has expired.
day_40 = trading_days[40]
print(f"\n--- Fast Forwarding to Day 40: {day_40.date()} ---")

# 1. Get the new state of the world
daily_data_40 = df_univ.loc[day_40].copy()
daily_data_40 = daily_data_40.replace([np.inf, -np.inf], np.nan)

#current market prices for benchmark constituents 
prices_cad_40 = dict(zip(daily_data_40['permno'], daily_data_40['prc_adj_cad']))


# 2. Mark-to-Market
total_equity_cad_40 = 0.0
for permno, pos in positions.items():
    shares = pos['shares']
    price_today = prices_cad_40.get(permno)
    
    if price_today is None or pd.isna(price_today):
        price_today = pos['acb_per_share']
        
    total_equity_cad_40 += shares * price_today



--- Fast Forwarding to Day 40: 2015-03-03 ---


In [207]:
#========================
# DIVIDENDS 
divs_cad_40 = dict(zip(daily_data_40['permno'], daily_data_40['divamt_net_cad']))
# initialize day 40 dividends 
daily_dividend_income = 0.0
# Loop through what we own and see if they paid us today
for permno, pos in positions.items():
    dps = divs_cad_40.get(permno, 0.0)
    if dps > 0:
        payout = pos['shares'] * dps
        cash_cad += payout  # Add to our global cash balance!
        daily_dividend_income += payout

if daily_dividend_income > 0:
    print(f"Collected ${daily_dividend_income:,.2f} in CAD dividends on {day_40.date()}.")


total_aum = total_equity_cad_40 + cash_cad
print(f"Day 40 Total AUM: ${total_aum:,.4f}")

Collected $6.86 in CAD dividends on 2015-03-03.
Day 40 Total AUM: $1,088,086.1608


In [208]:
# 3. THE HARVEST SCANNER (With CRA Logic)
harvest_threshold = -0.05
harvest_candidates = []

# First we update our do not harvest list 
do_not_harvest = [
    permno for permno, last_buy in last_buy_dates.items()
    if (day_40 - last_buy).days <= 30
]

dropped_more_than_threshold = 0

for permno, pos in positions.items():
    acb = pos['acb_per_share']
    price_today = prices_cad_40.get(permno)
    
    if price_today is None or pd.isna(price_today) or acb <= 0:
        continue
        
    return_pct = (price_today - acb) / acb
    
    if return_pct <= harvest_threshold:
        dropped_more_than_threshold += 1 
        # The CRA Backward Rule Check
        if permno not in do_not_harvest :
            harvest_candidates.append(permno)
print(f"Number of stocks down > 5%: {dropped_more_than_threshold}")
print(f"Number of stocks down > 5% that are legally harvestable today: {len(harvest_candidates)}")
if len(harvest_candidates) > 0:
    print(f"Sample permno candidates to harvest: {harvest_candidates[:5]}")

Number of stocks down > 5%: 20
Number of stocks down > 5% that are legally harvestable today: 20
Sample permno candidates to harvest: [11618, 13936, 21776, 21792, 23026]


In [209]:
list(positions.items())[:5]

[(10104, {'shares': 227.7116, 'acb_per_share': 51.976925}),
 (10107, {'shares': 426.1393, 'acb_per_share': 54.826100000000004}),
 (10138, {'shares': 13.5896, 'acb_per_share': 100.55360000000002}),
 (10145, {'shares': 40.5921, 'acb_per_share': 112.16021300555786}),
 (10516, {'shares': 33.3832, 'acb_per_share': 60.99345000000001})]

In [210]:
# DAY 40: PREPARING OPTIMIZER INPUTS
print("\n--- Prepping Data for the CVXPY Optimizer ---")

# 1. Get Day 40 Benchmark Weights
total_mkt_cap_40 = daily_data_40['mkt_cap_usd'].sum()
weights_40 = daily_data_40['mkt_cap_usd'] / total_mkt_cap_40
bench_w_40 = pd.Series(weights_40.values, index=daily_data_40['permno'])

# 2. Calculate our Current Portfolio Weights
current_w_40 = pd.Series({
    p: (pos['shares'] * prices_cad_40.get(p, pos['acb_per_share'])) / total_aum 
    for p, pos in positions.items()
})


--- Prepping Data for the CVXPY Optimizer ---


In [211]:
from b1_risk_model import FactorRiskModel
risk_model = FactorRiskModel()

# 3. Get the V Matrix (Union of Benchmark + Owned Stocks to prevent crashes)
owned_permnos = list(positions.keys())
benchmark_permnos_40 = daily_data_40['permno'].values
optimization_universe = np.unique(np.concatenate([benchmark_permnos_40, owned_permnos]))

V_mat_40,  X_aligned, F_t, D_sq = risk_model.build_factor_covariance(day_40, optimization_universe)

# 4. Clean up any expired forward lockouts 
# Since we have not sold any positions, superficial_loss_lockouts is empty in this demo 
do_not_buy =[p for p, exp_date in superficial_loss_lockouts.items() if exp_date >= day_40]



Loading Risk Model inputs from Phase A (Quant Infrastructure)...
Building daily return matrix for Ledoit-Wolf failsafe...
Loading Factor Exposures (X)...
Loading Factor Covariance Matrices (F)...
Loading Idiosyncratic Risk (Delta)...


In [212]:
X_aligned.columns

Index(['size', 'value', 'momentum', 'fin_constraint', 'profitability',
       'investment', 'volatility', 'Ind_Chemicals', 'Ind_Consumer',
       'Ind_Durables', 'Ind_Energy', 'Ind_Finance', 'Ind_Healthcare',
       'Ind_Other', 'Ind_Services', 'Ind_Shops', 'Ind_Technology',
       'Ind_Telecom', 'Ind_Utilities'],
      dtype='object')

In [213]:
F_t

,size,value,momentum,fin_constraint,profitability,investment,volatility,Ind_Chemicals,Ind_Consumer,Ind_Durables,Ind_Energy,Ind_Finance,Ind_Healthcare,Ind_Other,Ind_Services,Ind_Shops,Ind_Technology,Ind_Telecom,Ind_Utilities
size,1.132131e-04,1.536323e-06,-0.000023,5.884107e-05,6.336083e-07,8.166450e-06,0.000120,1.220752e-04,0.000104,1.192294e-04,0.000046,1.089594e-04,1.294370e-04,1.180416e-04,1.384284e-04,1.064535e-04,1.619957e-04,1.085780e-04,0.000167
value,1.536323e-06,5.340238e-06,-0.000002,1.329050e-07,9.022626e-08,1.031599e-07,0.000008,2.221410e-05,0.000018,2.476279e-05,0.000024,1.945560e-05,1.462715e-05,2.048194e-05,2.025206e-05,1.649236e-05,1.058146e-05,1.447512e-05,0.000022
momentum,-2.339307e-05,-1.915897e-06,0.000070,2.294594e-06,1.261360e-06,2.217821e-06,-0.000115,-2.031958e-04,-0.000156,-1.925739e-04,-0.000110,-1.957796e-04,-1.241830e-04,-1.510517e-04,-1.667625e-04,-1.662553e-04,-1.791224e-04,-1.562497e-04,-0.000112
fin_constraint,5.884107e-05,1.329050e-07,0.000002,8.513300e-05,-2.583388e-07,8.090685e-06,0.000040,-7.217660e-05,-0.000056,-4.174357e-05,-0.000043,-7.439170e-05,4.529326e-06,-2.354000e-05,-1.441118e-05,-6.198715e-05,5.095031e-06,-2.679165e-05,0.000044
profitability,6.336083e-07,9.022626e-08,0.000001,-2.583388e-07,2.265827e-06,1.235518e-07,-0.000003,5.232067e-07,0.000001,-7.310750e-07,0.000002,-2.638422e-08,-8.818126e-07,5.426284e-07,-9.954754e-07,-2.008598e-08,-2.065988e-07,3.560891e-06,-0.000004
investment,8.166450e-06,1.031599e-07,0.000002,8.090685e-06,1.235518e-07,7.781618e-06,0.000001,-1.369813e-05,-0.000012,-1.375662e-05,-0.000014,-1.708354e-05,1.831227e-06,-1.196828e-05,-6.482547e-06,-1.369764e-05,8.838499e-08,5.168038e-07,0.000003
volatility,1.200935e-04,7.877023e-06,-0.000115,3.994573e-05,-2.879048e-06,1.063694e-06,0.000475,6.877963e-04,0.000542,6.876161e-04,0.000562,6.250158e-04,5.047506e-04,5.661762e-04,5.954340e-04,5.766281e-04,6.717595e-04,5.202982e-04,0.000568
Ind_Chemicals,1.220752e-04,2.221410e-05,-0.000203,-7.217660e-05,5.232067e-07,-1.369813e-05,0.000688,3.502049e-03,0.002149,2.780739e-03,0.002439,2.191618e-03,1.548697e-03,2.264111e-03,2.261730e-03,2.233168e-03,2.322134e-03,2.059954e-03,0.001548
Ind_Consumer,1.035218e-04,1.813922e-05,-0.000156,-5.638670e-05,1.106129e-06,-1.199426e-05,0.000542,2.148859e-03,0.001763,2.008767e-03,0.001585,1.763424e-03,1.366621e-03,1.691592e-03,1.751759e-03,1.804175e-03,1.776707e-03,1.606300e-03,0.001323
Ind_Durables,1.192294e-04,2.476279e-05,-0.000193,-4.174357e-05,-7.310750e-07,-1.375662e-05,0.000688,2.780739e-03,0.002009,2.702808e-03,0.002243,2.151261e-03,1.556930e-03,2.176676e-03,2.208550e-03,2.159405e-03,2.284859e-03,2.015486e-03,0.001539


In [214]:
D_sq

permno
10104    0.002997
10107    0.003099
10138    0.002109
10145    0.001330
10147    0.003371
           ...   
92988    0.002400
93002    0.020194
93096    0.003271
93159    0.020194
93422    0.004108
Name: resid_vol, Length: 502, dtype: float64

In [215]:
np.min(D_sq)

0.0007127785005818895

### condition number of the covariance matrix 

In [216]:
eigenvalues_raw = np.linalg.eigvalsh(V_mat_40)
condition_number = eigenvalues_raw[-1] / eigenvalues_raw[0]
print(f"The smallest Eigenvalue for the return covariance matrix is {eigenvalues_raw[0]}")
print(f"The largest Eigenvalue for the return covariance matrix is {eigenvalues_raw[-1]}")
print(f"Raw Covariance Matrix Condition Number: {condition_number:,.2f}")

The smallest Eigenvalue for the return covariance matrix is 0.0007152920793948027
The largest Eigenvalue for the return covariance matrix is 0.8506976501682845
Raw Covariance Matrix Condition Number: 1,189.30


### Condition number is acceptable; we still perform a denoising procedure 

##  EigenDecomposing the Factor Covariance ($F_t$)
By running eigendecomposition on $F_t$, we look at those 19 tangled, overlapping factors and perfectly cleanly separate them into 19 mathematically orthogonal (independent) "Eigen-factors". TYhe orthogonality of eigenvectors of F is the result of **Spectral Theorem** They are going to be orthogonal becasue the F matrix is symmetric. 

In [217]:
eigenvalues_F, Q_F = np.linalg.eigh(F_t)

# sorting eigenvalues from largest to smallest 
sort_idx = np.argsort(eigenvalues_F)[::-1]
eigenvalues_F = eigenvalues_F[sort_idx]
# sorting eigenvectors from PC1 to PC19 
Q_F = Q_F[:, sort_idx]

lambda_F = np.diag(eigenvalues_F)

# Reconstructing F from eigenvectors and the eigenvalues  
F_reconstructed  = pd.DataFrame(data = Q_F @ lambda_F @ Q_F.T, 
                                index = F_t.index, 
                                columns = F_t.index)
is_perfect = np.allclose(F_t, F_reconstructed)
print(is_perfect)  

True


In [218]:
F_inv = pd.DataFrame(data = np.linalg.inv(F_t), 
                     index = F_t.index, 
                     columns= F_t.columns)
F_inv

,size,value,momentum,fin_constraint,profitability,investment,volatility,Ind_Chemicals,Ind_Consumer,Ind_Durables,Ind_Energy,Ind_Finance,Ind_Healthcare,Ind_Other,Ind_Services,Ind_Shops,Ind_Technology,Ind_Telecom,Ind_Utilities
size,21299.805572,858.054835,837.832518,-12036.273304,-13515.548306,-8701.949786,-3449.384272,-652.508649,-613.224005,2177.567283,469.615699,-380.550591,400.621856,-188.912897,-1222.896995,29.297509,-514.202979,484.158300,-721.193163
value,858.054835,213666.542642,852.781938,4.642060,-17853.495824,-11982.450086,-1803.312168,1305.809288,-1011.845346,-6030.717079,219.503093,259.947109,102.395576,162.600479,-3368.909644,1448.018752,5954.193660,945.339620,-1447.175264
momentum,837.832518,852.781938,27396.622667,-2096.582009,-10206.521457,-4812.976935,6751.942320,304.315213,969.695659,518.082365,-576.665834,1092.681368,-725.154063,-490.713178,-475.568480,-622.590376,-464.926030,1012.225862,-836.046414
fin_constraint,-12036.273304,4.642060,-2096.582009,25257.263693,2165.156556,-5651.737503,-846.233167,659.806738,2757.167882,-1331.406393,135.055580,1947.605528,-507.875010,-562.490375,-1477.897966,810.174073,-709.257642,267.087942,-691.770971
profitability,-13515.548306,-17853.495824,-10206.521457,2165.156556,480126.117857,-1457.258502,3503.978984,449.441803,-6367.990987,1948.665198,-1281.542221,-1202.155495,485.108053,-1642.260445,6712.017630,276.094421,574.991740,-5108.780607,4041.524885
investment,-8701.949786,-11982.450086,-4812.976935,-5651.737503,-1457.258502,158135.360284,1005.766998,-1260.588039,3153.372538,1707.929010,317.735758,1526.906790,-2328.353711,1044.433586,584.117656,327.758132,-2045.758977,-2486.780278,-486.365154
volatility,-3449.384272,-1803.312168,6751.942320,-846.233167,3503.978984,1005.766998,6828.738290,63.479503,696.491763,-227.843913,-296.971340,-486.860812,-489.383394,190.877526,247.302359,-338.783619,-805.239834,550.009968,-611.797251
Ind_Chemicals,-652.508649,1305.809288,304.315213,659.806738,449.441803,-1260.588039,63.479503,1837.245051,-1215.759223,-1549.280499,-128.791405,217.803118,331.901406,-155.148628,36.797958,211.428312,112.111386,63.409973,202.239607
Ind_Consumer,-613.224005,-1011.845346,969.695659,2757.167882,-6367.990987,3153.372538,696.491763,-1215.759223,9210.761611,-256.622554,-73.820373,-728.215526,-1708.401250,-184.110185,-702.715922,-2414.575884,-327.927815,-171.506522,-1179.199001
Ind_Durables,2177.567283,-6030.717079,518.082365,-1331.406393,1948.665198,1707.929010,-227.843913,-1549.280499,-256.622554,7992.462603,-707.363885,-853.966133,642.493777,-1512.276306,-1597.876204,-836.327520,-1749.020886,-86.328209,-287.492926


In [219]:
# reconstructing the inverse factor covariance matrix using the eigenvectors and eigenvalues 
lambda_inv = np.diag(1 / eigenvalues_F)
F_inv_reconstructed = Q_F @ lambda_inv @ Q_F.T
np.allclose(F_inv, F_inv_reconstructed)

True

In [220]:
eigenvalues_F

array([2.23998350e-02, 2.30467552e-03, 1.16409596e-03, 6.91196814e-04,
       5.25637110e-04, 4.14435512e-04, 3.50460354e-04, 3.43181354e-04,
       3.27810532e-04, 2.15916512e-04, 1.58574288e-04, 1.13614189e-04,
       9.84037084e-05, 7.61265531e-05, 3.40343111e-05, 2.76287304e-05,
       6.38508607e-06, 4.63952956e-06, 2.07344088e-06])

In [221]:
cond_F = eigenvalues_F[0] / eigenvalues_F[-1]
print(f"The smallest Eigenvalue for the return covariance matrix is {eigenvalues_F[-1]}")
print(f"The largest Eigenvalue for the return covariance matrix is {eigenvalues_F[0]}")
print(f"Raw Factor Covariance Matrix Condition Number: {cond_F:,.2f}")

The smallest Eigenvalue for the return covariance matrix is 2.0734408801643087e-06
The largest Eigenvalue for the return covariance matrix is 0.022399835000648877
Raw Factor Covariance Matrix Condition Number: 10,803.22


### Raw Covariance Matrix Condition Number of 10,803.22 is too high. This is becasue the smallest eigenvalue is too close to zero, i.e. due to noise in the data, there's a highly obscure, deeply buried linear combination of factors that histrocailly moved almost zero percent. The optimizer will see this and thinks it has discovered a riskless arbitrage spread between factors, and multiply its trades by 1/2.07e-6

In [222]:
eigenvalues_F = eigenvalues_F[::-1]
eigenvalues_F

array([2.07344088e-06, 4.63952956e-06, 6.38508607e-06, 2.76287304e-05,
       3.40343111e-05, 7.61265531e-05, 9.84037084e-05, 1.13614189e-04,
       1.58574288e-04, 2.15916512e-04, 3.27810532e-04, 3.43181354e-04,
       3.50460354e-04, 4.14435512e-04, 5.25637110e-04, 6.91196814e-04,
       1.16409596e-03, 2.30467552e-03, 2.23998350e-02])

In [223]:
F_t

,size,value,momentum,fin_constraint,profitability,investment,volatility,Ind_Chemicals,Ind_Consumer,Ind_Durables,Ind_Energy,Ind_Finance,Ind_Healthcare,Ind_Other,Ind_Services,Ind_Shops,Ind_Technology,Ind_Telecom,Ind_Utilities
size,1.132131e-04,1.536323e-06,-0.000023,5.884107e-05,6.336083e-07,8.166450e-06,0.000120,1.220752e-04,0.000104,1.192294e-04,0.000046,1.089594e-04,1.294370e-04,1.180416e-04,1.384284e-04,1.064535e-04,1.619957e-04,1.085780e-04,0.000167
value,1.536323e-06,5.340238e-06,-0.000002,1.329050e-07,9.022626e-08,1.031599e-07,0.000008,2.221410e-05,0.000018,2.476279e-05,0.000024,1.945560e-05,1.462715e-05,2.048194e-05,2.025206e-05,1.649236e-05,1.058146e-05,1.447512e-05,0.000022
momentum,-2.339307e-05,-1.915897e-06,0.000070,2.294594e-06,1.261360e-06,2.217821e-06,-0.000115,-2.031958e-04,-0.000156,-1.925739e-04,-0.000110,-1.957796e-04,-1.241830e-04,-1.510517e-04,-1.667625e-04,-1.662553e-04,-1.791224e-04,-1.562497e-04,-0.000112
fin_constraint,5.884107e-05,1.329050e-07,0.000002,8.513300e-05,-2.583388e-07,8.090685e-06,0.000040,-7.217660e-05,-0.000056,-4.174357e-05,-0.000043,-7.439170e-05,4.529326e-06,-2.354000e-05,-1.441118e-05,-6.198715e-05,5.095031e-06,-2.679165e-05,0.000044
profitability,6.336083e-07,9.022626e-08,0.000001,-2.583388e-07,2.265827e-06,1.235518e-07,-0.000003,5.232067e-07,0.000001,-7.310750e-07,0.000002,-2.638422e-08,-8.818126e-07,5.426284e-07,-9.954754e-07,-2.008598e-08,-2.065988e-07,3.560891e-06,-0.000004
investment,8.166450e-06,1.031599e-07,0.000002,8.090685e-06,1.235518e-07,7.781618e-06,0.000001,-1.369813e-05,-0.000012,-1.375662e-05,-0.000014,-1.708354e-05,1.831227e-06,-1.196828e-05,-6.482547e-06,-1.369764e-05,8.838499e-08,5.168038e-07,0.000003
volatility,1.200935e-04,7.877023e-06,-0.000115,3.994573e-05,-2.879048e-06,1.063694e-06,0.000475,6.877963e-04,0.000542,6.876161e-04,0.000562,6.250158e-04,5.047506e-04,5.661762e-04,5.954340e-04,5.766281e-04,6.717595e-04,5.202982e-04,0.000568
Ind_Chemicals,1.220752e-04,2.221410e-05,-0.000203,-7.217660e-05,5.232067e-07,-1.369813e-05,0.000688,3.502049e-03,0.002149,2.780739e-03,0.002439,2.191618e-03,1.548697e-03,2.264111e-03,2.261730e-03,2.233168e-03,2.322134e-03,2.059954e-03,0.001548
Ind_Consumer,1.035218e-04,1.813922e-05,-0.000156,-5.638670e-05,1.106129e-06,-1.199426e-05,0.000542,2.148859e-03,0.001763,2.008767e-03,0.001585,1.763424e-03,1.366621e-03,1.691592e-03,1.751759e-03,1.804175e-03,1.776707e-03,1.606300e-03,0.001323
Ind_Durables,1.192294e-04,2.476279e-05,-0.000193,-4.174357e-05,-7.310750e-07,-1.375662e-05,0.000688,2.780739e-03,0.002009,2.702808e-03,0.002243,2.151261e-03,1.556930e-03,2.176676e-03,2.208550e-03,2.159405e-03,2.284859e-03,2.015486e-03,0.001539


In [224]:
# Reconstructing F from eigenvectors and the eigenvalues  
F_reconstructed  = pd.DataFrame(data = Q_F @ lambda_F @ Q_F.T, 
                                index = F_t.index, 
                                columns = F_t.index)

In [225]:
F_reconstructed

,size,value,momentum,fin_constraint,profitability,investment,volatility,Ind_Chemicals,Ind_Consumer,Ind_Durables,Ind_Energy,Ind_Finance,Ind_Healthcare,Ind_Other,Ind_Services,Ind_Shops,Ind_Technology,Ind_Telecom,Ind_Utilities
size,1.132131e-04,1.536323e-06,-0.000023,5.884107e-05,6.336083e-07,8.166450e-06,0.000120,1.220752e-04,0.000104,1.192294e-04,0.000046,1.089594e-04,1.294370e-04,1.180416e-04,1.384284e-04,1.064535e-04,1.619957e-04,1.085780e-04,0.000167
value,1.536323e-06,5.340238e-06,-0.000002,1.329050e-07,9.022626e-08,1.031599e-07,0.000008,2.221410e-05,0.000018,2.476279e-05,0.000024,1.945560e-05,1.462715e-05,2.048194e-05,2.025206e-05,1.649236e-05,1.058146e-05,1.447512e-05,0.000022
momentum,-2.339307e-05,-1.915897e-06,0.000070,2.294594e-06,1.261360e-06,2.217821e-06,-0.000115,-2.031958e-04,-0.000156,-1.925739e-04,-0.000110,-1.957796e-04,-1.241830e-04,-1.510517e-04,-1.667625e-04,-1.662553e-04,-1.791224e-04,-1.562497e-04,-0.000112
fin_constraint,5.884107e-05,1.329050e-07,0.000002,8.513300e-05,-2.583388e-07,8.090685e-06,0.000040,-7.217660e-05,-0.000056,-4.174357e-05,-0.000043,-7.439170e-05,4.529326e-06,-2.354000e-05,-1.441118e-05,-6.198715e-05,5.095031e-06,-2.679165e-05,0.000044
profitability,6.336083e-07,9.022626e-08,0.000001,-2.583388e-07,2.265827e-06,1.235518e-07,-0.000003,5.232067e-07,0.000001,-7.310750e-07,0.000002,-2.638422e-08,-8.818126e-07,5.426284e-07,-9.954754e-07,-2.008598e-08,-2.065988e-07,3.560891e-06,-0.000004
investment,8.166450e-06,1.031599e-07,0.000002,8.090685e-06,1.235518e-07,7.781618e-06,0.000001,-1.369813e-05,-0.000012,-1.375662e-05,-0.000014,-1.708354e-05,1.831227e-06,-1.196828e-05,-6.482547e-06,-1.369764e-05,8.838499e-08,5.168038e-07,0.000003
volatility,1.200935e-04,7.877023e-06,-0.000115,3.994573e-05,-2.879048e-06,1.063694e-06,0.000475,6.877963e-04,0.000542,6.876161e-04,0.000562,6.250158e-04,5.047506e-04,5.661762e-04,5.954340e-04,5.766281e-04,6.717595e-04,5.202982e-04,0.000568
Ind_Chemicals,1.220752e-04,2.221410e-05,-0.000203,-7.217660e-05,5.232067e-07,-1.369813e-05,0.000688,3.502049e-03,0.002149,2.780739e-03,0.002439,2.191618e-03,1.548697e-03,2.264111e-03,2.261730e-03,2.233168e-03,2.322134e-03,2.059954e-03,0.001548
Ind_Consumer,1.035218e-04,1.813922e-05,-0.000156,-5.638670e-05,1.106129e-06,-1.199426e-05,0.000542,2.148859e-03,0.001763,2.008767e-03,0.001585,1.763424e-03,1.366621e-03,1.691592e-03,1.751759e-03,1.804175e-03,1.776707e-03,1.606300e-03,0.001323
Ind_Durables,1.192294e-04,2.476279e-05,-0.000193,-4.174357e-05,-7.310750e-07,-1.375662e-05,0.000688,2.780739e-03,0.002009,2.702808e-03,0.002243,2.151261e-03,1.556930e-03,2.176676e-03,2.208550e-03,2.159405e-03,2.284859e-03,2.015486e-03,0.001539


In [226]:
print("--- F_t Eigenvalues (Top 5) ---")
print(np.round(eigenvalues_F[:5], 6))
print("--- F_t Eigenvalues (Bottom 5) ---")
print(np.round(eigenvalues_F[-5:], 6))

--- F_t Eigenvalues (Top 5) ---
[2.0e-06 5.0e-06 6.0e-06 2.8e-05 3.4e-05]
--- F_t Eigenvalues (Bottom 5) ---
[0.000526 0.000691 0.001164 0.002305 0.0224  ]


### Too low or too high cannot be subjetive. we are going to utilize Marchenko-Pastur to find reasonable bound for our eigenvalues 

In [227]:
from scipy.optimize import minimize 
from sklearn.neighbors import KernelDensity 

In [228]:
# We need the time T we used to estimate F. since I didnt use simple moving average and used EWMA of 36 months on factor return, I cannot use 36. 
# calculating Effective Time for our F matrix 
halflife = 36
alpha = 1 - np.exp(-np.log(2) / halflife)
T_eff = 1 / alpha # months because our original Fam-mcBeth regressions were monthly 
N_factors = len(F_t)
q = T_eff / N_factors
print(f"System T/N Ratio (q): {q:.2f}")

System T/N Ratio (q): 2.76


In [229]:
np.min(eigenvalues_F)

2.0734408801643087e-06

In [230]:
# We take our real eigenvalues and draw a smoothed histogram over them.

obs = eigenvalues_F.reshape(-1,1)
# find the kde of the eigenvalues 
kde = KernelDensity(kernel = 'gaussian', bandwidth= 0.01).fit(obs)
# Create a grid of x-values to evaluate the curve
x_grid = np.linspace(np.min(eigenvalues_F) - 0.1, np.max(eigenvalues_F) + 0.1, 1000)
# now populate the kde i.e. trace out the pdf for the x axis 
log_prob = kde.score_samples(x_grid.reshape(-1,1))
# transform to normal probabilites 
empirical_pdf = np.exp(log_prob)

In [231]:
# for a given set of variance arrays, the function 
# 1. calcualtes theoretical MP bounds, 
# 2. calcualtes the theoretical pdf
# 3. calculates sse 

def objective_function(var_array):
    var = var_array[0] 
    
    # Calculate the theoretical Marchenko-Pastur bounds for this guess
    eMin = var * (1 - (1./q)**0.5)**2
    eMax = var * (1 + (1./q)**0.5)**2
    
    # Calculate the theoretical MP curve
    theoretical_pdf = np.zeros_like(x_grid)
    valid_idx = (x_grid >= eMin) & (x_grid <= eMax)
    
    if np.any(valid_idx):
        valid_x = x_grid[valid_idx]
        theoretical_pdf[valid_idx] = (q / (2 * np.pi * var * valid_x)) * np.sqrt((eMax - valid_x) * (valid_x - eMin))
    
    # Calculate the Error: Sum of Squared Differences between Theory and Reality
    sse = np.sum((theoretical_pdf - empirical_pdf)**2)
    return sse

In [232]:
# Give the optimizer a starting guess and bound it so variance can't be negative
initial_guess = [np.var(eigenvalues_F) / 2.0]
bounds = [(1e-5, 1 - 1e-5)]

print("Optimizing Marchenko-Pastur fit...")
result = minimize(objective_function, initial_guess, bounds=bounds)

# true variance of the noise 
true_var = result.x[0]
# the upper MP bound, below which are our noise eigenvectors 
lambda_plus = true_var * (1 + (1./q)**0.5)**2

print(f"Optimized Noise Variance (Sigma^2): {true_var:.6f}")
print(f"De Prado Cutoff (Lambda+): {lambda_plus:.6f}")


Optimizing Marchenko-Pastur fit...
Optimized Noise Variance (Sigma^2): 0.000012
De Prado Cutoff (Lambda+): 0.000031


In [233]:
eigenvalues_F

array([2.07344088e-06, 4.63952956e-06, 6.38508607e-06, 2.76287304e-05,
       3.40343111e-05, 7.61265531e-05, 9.84037084e-05, 1.13614189e-04,
       1.58574288e-04, 2.15916512e-04, 3.27810532e-04, 3.43181354e-04,
       3.50460354e-04, 4.14435512e-04, 5.25637110e-04, 6.91196814e-04,
       1.16409596e-03, 2.30467552e-03, 2.23998350e-02])

In [235]:
# Identify which factors are real, and which are noise
is_signal = eigenvalues_F > lambda_plus
is_noise = eigenvalues_F <= lambda_plus

print(f"Number of True Signals found: {np.sum(is_signal)} out of {N_factors}")

Number of True Signals found: 15 out of 19


In [236]:
# We do not throw the noise away. We calculate the average variance of the 
# noisy eigenvalues, and overwrite all of them with that average.
# This flattens the noise floor while preserving total matrix variance.

average_noise_variance = np.mean(eigenvalues_F[is_noise])
denoised_eigenvalues_F = eigenvalues_F.copy()
denoised_eigenvalues_F[is_noise] = average_noise_variance

print(f"Original smallest eigenvalue: {eigenvalues_F[0]:.8f}")
print(f"New constant noise floor: {average_noise_variance:.8f}")

Original smallest eigenvalue: 0.00000207
New constant noise floor: 0.00001018


In [238]:
# Build the diagonal matrix with our new, flattened eigenvalues
lambda_F_cleaned = np.diag(denoised_eigenvalues_F)

F_t_denoised = pd.DataFrame(data = Q_F @ lambda_F_cleaned @ Q_F.T, 
                            index = F_t.index,
                            columns = F_t.columns)
F_t_denoised


,size,value,momentum,fin_constraint,profitability,investment,volatility,Ind_Chemicals,Ind_Consumer,Ind_Durables,Ind_Energy,Ind_Finance,Ind_Healthcare,Ind_Other,Ind_Services,Ind_Shops,Ind_Technology,Ind_Telecom,Ind_Utilities
size,0.000496,0.000050,4.111721e-05,-0.000168,-0.000643,-0.000048,-8.258970e-05,-0.000011,1.288548e-05,0.000028,1.704851e-05,0.000011,-0.000005,-2.835324e-06,-0.000029,8.214262e-06,-2.676980e-05,0.000016,-0.000031
value,0.000050,0.002339,3.904011e-05,-0.000002,-0.001339,-0.000232,-2.697838e-05,0.000012,3.348758e-06,-0.000063,4.455326e-06,0.000003,0.000001,2.772437e-06,-0.000049,1.257461e-05,5.956521e-05,0.000024,-0.000025
momentum,0.000041,0.000039,5.112828e-04,-0.000030,-0.000491,-0.000025,8.511610e-05,0.000007,1.204327e-05,0.000007,-7.898544e-06,0.000026,-0.000010,-1.224364e-05,-0.000008,-6.189296e-06,-4.957832e-07,0.000011,-0.000013
fin_constraint,-0.000168,-0.000002,-3.031810e-05,0.000540,0.000115,-0.000037,-3.808212e-05,0.000012,4.965468e-05,-0.000014,6.067593e-06,0.000049,-0.000014,-1.637701e-05,-0.000023,2.310922e-05,-3.018641e-05,0.000002,-0.000020
profitability,-0.000643,-0.001339,-4.908743e-04,0.000115,0.022264,-0.000039,1.644962e-04,0.000019,-2.931644e-04,0.000102,-5.972414e-05,-0.000056,0.000023,-7.590491e-05,0.000320,1.035169e-05,1.290275e-05,-0.000240,0.000192
investment,-0.000048,-0.000232,-2.495782e-05,-0.000037,-0.000039,0.001208,4.486541e-06,-0.000009,2.178801e-05,0.000016,3.310573e-06,0.000014,-0.000016,7.214100e-06,0.000004,4.401117e-06,-2.109856e-05,-0.000018,-0.000005
volatility,-0.000083,-0.000027,8.511610e-05,-0.000038,0.000164,0.000004,2.125057e-04,0.000002,1.380410e-05,-0.000009,-5.170157e-06,-0.000024,-0.000013,1.834895e-05,-0.000005,9.273937e-08,-4.097511e-05,0.000033,-0.000026
Ind_Chemicals,-0.000011,0.000012,6.842379e-06,0.000012,0.000019,-0.000009,1.583494e-06,0.000042,-4.004945e-05,-0.000058,5.458425e-06,0.000014,0.000022,-5.062904e-06,0.000004,1.199516e-05,7.956311e-06,0.000010,0.000002
Ind_Consumer,0.000013,0.000003,1.204327e-05,0.000050,-0.000293,0.000022,1.380410e-05,-0.000040,3.095255e-04,-0.000016,1.698342e-06,-0.000037,-0.000046,-5.938979e-06,-0.000030,-7.574959e-05,-6.661665e-08,-0.000002,-0.000042
Ind_Durables,0.000028,-0.000063,7.176840e-06,-0.000014,0.000102,0.000016,-9.116959e-06,-0.000058,-1.573844e-05,0.000297,-2.813407e-05,-0.000028,0.000020,-6.274242e-05,-0.000037,-3.799187e-05,-5.776441e-05,-0.000008,-0.000002


## shrinkage 

In [ ]:
# we are going to use the same threshold for lamda to disnguish signal from noise 
# creating boolean masks to separate the factors
# is_signal = eigenvalues_F > lambda_plus
# is_noise = eigenvalues_F <= lambda_plus

In [241]:
is_noise

array([ True,  True,  True,  True, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False])

In [245]:
# we are going to filter the original eigenvector matrix 
# filtring to get the signal eigenvectors (eigenvectors with lambda higher than lambda_plus)
# the untouched signal vault 
W_L = Q_F[:, is_signal]
lambda_L = np.diag(eigenvalues_F[is_signal])

# we are going to filter the original eigenvector matrix 
# filtring to get the noisy eigenvectors (eigenvectors with lambda lower than lambda_plus)
#  R denotes random
W_R = Q_F[:, is_noise]
lambda_R = np.diag(eigenvalues_F[is_noise])

# Rebuild the Covariance of just the true macroeconomic signals
C_signal = W_L @ lambda_L @ W_L.T

# Rebuild the Covariance of the raw, hallucinated noise
C_noise_raw = W_R @ lambda_R @ W_R.T

In [253]:
# Diagonalize the noise 
# Create a diagonal matrix of the C_noise_raw. Keeping only its diagonal elements 
C_noise_diag = np.diag(np.diag(C_noise_raw))


In [259]:
# alpha is the mixing dial. 
# alpha = 1.0 means keep all fake correlations (dangerous).
# alpha = 0.0 means perfect diagonal shrinkage (safest).
alpha = 0.1
F_targeted = pd.DataFrame(data = C_signal + (alpha * C_noise_raw) + ((1 - alpha) * C_noise_diag),
                          index = F_t.index, 
                          columns = F_t.columns)
F_targeted


,size,value,momentum,fin_constraint,profitability,investment,volatility,Ind_Chemicals,Ind_Consumer,Ind_Durables,Ind_Energy,Ind_Finance,Ind_Healthcare,Ind_Other,Ind_Services,Ind_Shops,Ind_Technology,Ind_Telecom,Ind_Utilities
size,0.000496,0.000050,4.113787e-05,-0.000168,-0.000643,-0.000048,-8.278908e-05,-0.000011,1.275500e-05,0.000028,1.733337e-05,0.000011,-5.558277e-06,-2.867155e-06,-0.000029,8.156486e-06,-2.681047e-05,1.551962e-05,-0.000031
value,0.000050,0.002339,3.904096e-05,-0.000002,-0.001339,-0.000232,-2.699240e-05,0.000012,3.334215e-06,-0.000063,4.440280e-06,0.000003,1.223845e-06,2.760223e-06,-0.000049,1.257381e-05,5.958406e-05,2.415913e-05,-0.000025
momentum,0.000041,0.000039,5.112737e-04,-0.000030,-0.000491,-0.000025,8.515892e-05,0.000007,1.214827e-05,0.000007,-8.092646e-06,0.000026,-9.805508e-06,-1.215015e-05,-0.000007,-6.056188e-06,-4.025185e-07,1.111142e-05,-0.000013
fin_constraint,-0.000168,-0.000002,-3.031603e-05,0.000540,0.000115,-0.000037,-3.821428e-05,0.000013,4.965671e-05,-0.000014,6.067086e-06,0.000049,-1.412392e-05,-1.629371e-05,-0.000023,2.318670e-05,-3.021970e-05,1.647603e-06,-0.000021
profitability,-0.000643,-0.001339,-4.908749e-04,0.000115,0.022264,-0.000039,1.645028e-04,0.000019,-2.931590e-04,0.000102,-5.973854e-05,-0.000056,2.254030e-05,-7.590204e-05,0.000320,1.035195e-05,1.289151e-05,-2.396616e-04,0.000192
investment,-0.000048,-0.000232,-2.495750e-05,-0.000037,-0.000039,0.001208,4.468034e-06,-0.000009,2.179319e-05,0.000016,3.317930e-06,0.000014,-1.635204e-05,7.233531e-06,0.000004,4.412013e-06,-2.112108e-05,-1.815968e-05,-0.000005
volatility,-0.000083,-0.000027,8.515892e-05,-0.000038,0.000165,0.000004,2.123181e-04,0.000002,1.344073e-05,-0.000009,-5.110489e-06,-0.000025,-1.349481e-05,1.813142e-05,-0.000005,-1.209142e-07,-4.117770e-05,3.240206e-05,-0.000028
Ind_Chemicals,-0.000011,0.000012,6.924438e-06,0.000013,0.000019,-0.000009,1.881722e-06,0.000047,-4.125410e-05,-0.000060,4.663958e-06,0.000013,2.312562e-05,-6.849492e-06,0.000004,1.066689e-05,8.394622e-06,1.136238e-05,0.000003
Ind_Consumer,0.000013,0.000003,1.214827e-05,0.000050,-0.000293,0.000022,1.344073e-05,-0.000041,3.090711e-04,-0.000017,1.959807e-06,-0.000038,-4.648756e-05,-6.841328e-06,-0.000031,-7.672860e-05,-7.182518e-07,-2.609292e-06,-0.000043
Ind_Durables,0.000028,-0.000063,7.257198e-06,-0.000014,0.000102,0.000016,-9.211878e-06,-0.000060,-1.662583e-05,0.000296,-2.911400e-05,-0.000029,1.980486e-05,-6.381017e-05,-0.000038,-3.905873e-05,-5.871251e-05,-8.974551e-06,-0.000002


In [260]:
np.allclose(F_t, F_targeted)

False

## Ledoit Wolf on the noise 

In [285]:
from sklearn.covariance import LedoitWolf

In [261]:
factor_returns = pd.read_parquet(DATA_DIR / 'factor_returns.parquet')

In [264]:
factor_returns.index

DatetimeIndex(['1990-01-31', '1990-02-28', '1990-03-31', '1990-04-30',
               '1990-05-31', '1990-06-30', '1990-07-31', '1990-08-31',
               '1990-09-30', '1990-10-31',
               ...
               '2023-03-31', '2023-04-30', '2023-05-31', '2023-06-30',
               '2023-07-31', '2023-08-31', '2023-09-30', '2023-10-31',
               '2023-11-30', '2023-12-31'],
              dtype='datetime64[ns]', name='date', length=408, freq=None)

In [279]:
target_date  = pd.to_datetime(day_40)
lookback_months = T_eff
factor_rets_slice = factor_returns.loc[:target_date].tail(np.floor(lookback_months).astype(int))
factor_rets_slice

,size,value,momentum,fin_constraint,profitability,investment,volatility,Ind_Chemicals,Ind_Consumer,Ind_Durables,Ind_Energy,Ind_Finance,Ind_Healthcare,Ind_Other,Ind_Services,Ind_Shops,Ind_Technology,Ind_Telecom,Ind_Utilities
date,,,,,,,,,,,,,,,,,,,
2010-11-30,0.000408,-0.001941,0.007627,0.009111,0.000324,-0.000341,0.005493,0.016160,-0.005211,0.026636,0.072802,-0.004769,-0.007282,0.048555,0.001963,0.047898,0.001890,-0.015998,-0.007307
2010-12-31,0.016076,0.002485,-0.013971,0.008197,0.000776,-0.002141,0.033423,0.054495,0.062679,0.087707,0.091186,0.102883,0.074840,0.084687,0.061216,0.050640,0.054259,0.066374,0.068597
2011-01-31,0.010983,-0.001275,-0.008053,0.006820,0.003025,-0.001298,0.006801,0.010051,0.004002,0.038200,0.071240,0.022059,0.001605,0.002885,0.010007,-0.005780,0.035155,0.016088,0.033290
2011-02-28,0.015108,0.002707,-0.000119,0.019655,0.000140,-0.000301,0.016501,0.050184,0.041872,0.021828,0.083763,0.035394,0.050008,0.003284,0.062421,0.036742,0.031201,0.055448,0.041935
2011-03-31,0.000744,-0.000374,0.006816,0.010034,-0.000327,0.005705,0.001249,0.074703,0.026065,0.026363,0.022095,-0.001251,0.034908,0.025592,-0.003510,0.011065,-0.016381,0.006573,0.016004
2011-04-30,0.018309,-0.001672,-0.000682,0.014947,0.000304,0.002509,0.000663,0.087833,0.038014,0.023227,0.006888,0.026114,0.062483,0.024524,0.028511,0.047310,0.030491,0.048559,0.049124
2011-05-31,0.005082,-0.001161,-0.001933,0.003301,-0.000181,0.000946,-0.002916,-0.008450,0.016482,-0.019747,-0.037840,-0.017222,0.020369,-0.017990,-0.008815,0.011870,-0.016319,0.025636,0.012867
2011-06-30,0.005018,-0.000450,0.006239,0.007483,0.001619,0.000715,-0.007186,0.002493,-0.014942,0.001910,-0.026049,-0.014025,-0.024332,-0.022014,-0.010059,-0.006075,-0.028962,-0.013710,-0.006096
2011-07-31,0.004969,-0.000087,0.000533,0.002073,0.000798,0.000544,0.002656,-0.009534,-0.014502,-0.052566,0.032405,-0.029318,-0.030452,-0.037661,-0.011964,-0.012916,-0.040157,-0.046077,-0.003691


In [283]:
# We will still use the noise eigenvectors we got from Marchenko and Pastur 
# We multiply our historical returns by the Noise Eigenvectors (W_R).
# Since W_R are the eigenvectors of the F matrix, each is a vector of factor weight 
# As a result multiplying factor returns by Noise eigenvectors 
# gives us the literal daily/monthly returns of the "Noise Portfolios".
noise_returns = factor_returns @ W_R
noise_returns

,0,1,2,3
date,,,,
1990-01-31,-0.239131,0.028557,-0.018770,0.044617
1990-02-28,0.052624,0.010269,-0.013605,-0.002510
1990-03-31,0.094337,-0.028061,-0.012662,-0.011322
1990-04-30,-0.113329,-0.023040,-0.008553,-0.018760
1990-05-31,0.314052,-0.045849,0.030919,-0.045325
...,...,...,...,...
2023-08-31,-0.029435,0.036347,-0.013236,-0.045558
2023-09-30,-0.131550,0.053349,0.001224,-0.022182
2023-10-31,-0.083922,-0.001405,0.060301,-0.028112


In [286]:
lw = LedoitWolf()
lw.fit(noise_returns)


LedoitWolf()

In [287]:
# Sklearn's 'shrinkage_' attribute represents the weight given to the diagonal target.
# In De Prado's formula, this is exactly equal to (1 - alpha).
optimal_1_minus_alpha = lw.shrinkage_
optimal_alpha = 1.0 - optimal_1_minus_alpha

print(f"Ledoit-Wolf calculated the optimal Noise Alpha as: {optimal_alpha:.4f}")

Ledoit-Wolf calculated the optimal Noise Alpha as: 0.9864


In [288]:
F_targeted = pd.DataFrame(data = C_signal + (optimal_alpha * C_noise_raw) + ((1 - optimal_alpha) * C_noise_diag),
                          index = F_t.index, 
                          columns = F_t.columns)
F_targeted

,size,value,momentum,fin_constraint,profitability,investment,volatility,Ind_Chemicals,Ind_Consumer,Ind_Durables,Ind_Energy,Ind_Finance,Ind_Healthcare,Ind_Other,Ind_Services,Ind_Shops,Ind_Technology,Ind_Telecom,Ind_Utilities
size,0.000496,0.000050,4.112814e-05,-0.000168,-0.000643,-0.000048,-8.267744e-05,-0.000011,0.000013,0.000028,1.716065e-05,0.000011,-0.000005,-0.000003,-0.000030,8.162312e-06,-2.686128e-05,1.546032e-05,-0.000031
value,0.000050,0.002339,3.903933e-05,-0.000002,-0.001339,-0.000232,-2.698250e-05,0.000012,0.000003,-0.000063,4.403026e-06,0.000003,0.000001,0.000003,-0.000050,1.257863e-05,5.951248e-05,2.405851e-05,-0.000025
momentum,0.000041,0.000039,5.112737e-04,-0.000030,-0.000491,-0.000025,8.514207e-05,0.000007,0.000012,0.000007,-7.953145e-06,0.000026,-0.000010,-0.000012,-0.000007,-6.101460e-06,-3.655523e-07,1.118280e-05,-0.000013
fin_constraint,-0.000168,-0.000002,-3.031265e-05,0.000540,0.000115,-0.000037,-3.814420e-05,0.000012,0.000050,-0.000014,6.225056e-06,0.000049,-0.000014,-0.000016,-0.000023,2.311959e-05,-2.997028e-05,2.035277e-06,-0.000021
profitability,-0.000643,-0.001339,-4.908739e-04,0.000115,0.022264,-0.000039,1.644976e-04,0.000018,-0.000293,0.000102,-5.971174e-05,-0.000056,0.000023,-0.000076,0.000320,1.034863e-05,1.292693e-05,-2.396115e-04,0.000192
investment,-0.000048,-0.000232,-2.495595e-05,-0.000037,-0.000039,0.001208,4.475883e-06,-0.000009,0.000022,0.000016,3.370924e-06,0.000014,-0.000016,0.000007,0.000004,4.397063e-06,-2.102825e-05,-1.802252e-05,-0.000005
volatility,-0.000083,-0.000027,8.514207e-05,-0.000038,0.000164,0.000004,2.123181e-04,0.000002,0.000014,-0.000009,-5.291911e-06,-0.000024,-0.000013,0.000018,-0.000005,-1.000550e-07,-4.128688e-05,3.226265e-05,-0.000027
Ind_Chemicals,-0.000011,0.000012,6.811684e-06,0.000012,0.000018,-0.000009,1.765725e-06,0.000047,-0.000040,-0.000059,1.689067e-06,0.000013,0.000021,-0.000005,0.000002,1.157364e-05,3.398997e-06,3.960196e-06,0.000006
Ind_Consumer,0.000013,0.000003,1.210202e-05,0.000050,-0.000293,0.000022,1.358229e-05,-0.000040,0.000309,-0.000016,1.133645e-06,-0.000037,-0.000047,-0.000006,-0.000031,-7.640975e-05,-1.413151e-06,-3.717879e-06,-0.000042
Ind_Durables,0.000028,-0.000063,7.236950e-06,-0.000014,0.000102,0.000016,-9.269596e-06,-0.000059,-0.000016,0.000296,-2.916965e-05,-0.000028,0.000020,-0.000063,-0.000038,-3.877995e-05,-5.889940e-05,-9.388256e-06,-0.000002


In [ ]:

import cvxpy as cp
# ==========================================
# DAY 40: RUNNING THE OPTIMIZER
# ==========================================

# Align all vectors to the V_matrix index to ensure perfect linear algebra
permnos = V_mat_40.index.values
# Number of assets 
N = len(permnos)
# current portfolio and benchmark weights 
h_current = current_w_40.reindex(permnos).fillna(0.0).values
h_b = bench_w_40.reindex(permnos).fillna(0.0).values
V = V_mat_40.values
print('the risk model for the current period is ready!')

the risk model for the current period is ready!


In [231]:
# ==========================================
# BUILD THE OPPORTUNITY COST VECTOR
# ==========================================
# Initialize the penalty vector (Length N, matching V_matrix)
# The deeper the unrealized loss, the higher the opportunity cost (penalty) 
# of holding the asset and failing to exercise the tax-harvesting option.
harvest_opportunity_cost = np.zeros(N)

# for all the permnos in the V matrix, we are going to calculate the penalty vector 

# We only calculate penalties for the valid candidates we found earlier!
for i, permno in enumerate(permnos):
    if permno in harvest_candidates:
        acb = positions[permno]['acb_per_share']

        # get current price 
        price = prices_cad_40.get(permno, acb) 

        #calculate return from ACB
        return_pct = (price - acb) / acb
        
        # A -30% loss becomes a +0.30 opportunity cost.
        # CVXPY will minimize this, pushing the weight of this stock to 0.
        harvest_opportunity_cost[i] = abs(return_pct)

harvesting_candidates = pd.Series(harvest_opportunity_cost, index = permnos, name = 'harvest_oc')
harvesting_candidates = harvesting_candidates[harvesting_candidates != 0.0]

harvesting_candidates_permnos = harvesting_candidates.index
print(f"Opportunity cost vector is ready. There are {len(harvesting_candidates_permnos)}")
print(f"Here are the harvesting candidate permnos on {day_40.date()} \n {list(harvesting_candidates_permnos)}")

Opportunity cost vector is ready. There are 25
Here are the harvesting candidate permnos on 2015-03-03 
 [11618, 13936, 15553, 21776, 21792, 23026, 24010, 27828, 39538, 53613, 59176, 75100, 78877, 79089, 81774, 82298, 82618, 83435, 84032, 85072, 89004, 89641, 90071, 90162, 93159]


In [232]:
harvesting_candidates

11618    0.079296
13936    0.110544
15553    0.063145
21776    0.066762
21792    0.066233
23026    0.069270
24010    0.062013
27828    0.087784
39538    0.062575
53613    0.093698
59176    0.064988
75100    0.104839
78877    0.129468
79089    0.177134
81774    0.053593
82298    0.134961
82618    0.142006
83435    0.097960
84032    0.117327
85072    0.211715
89004    0.095661
89641    0.053968
90071    0.067903
90162    0.054049
93159    0.150392
Name: harvest_oc, dtype: float64

In [233]:
# setting the optimization hyperparams 

max_tracking_error=0.005 
# this translates to tracking error voaltility of sqrt(0.005) = 7.07%

turnover_penalty=0.01

# inititializing the optimization variable (portfolio weights)
h = cp.Variable(N)

# ==========================================
# BUILD THE OBJECTIVE AND CONSTRAINTS
# ==========================================        

# Opt. OBJECTIVE: Minimize (Holding Penalty + Tracking Error + Friction)

# Note: Since opportunity_cost_vector is positive for losers, 
# we MINIMIZE (h^T * opportunity_cost_vector).
# By minimizing a positive product, CVXPY forces h -> 0 for the losers.
tax_penalty = h.T @ harvest_opportunity_cost
turnover = cp.norm1(h - h_current)

objective = cp.Minimize(tax_penalty + (turnover_penalty * turnover))

#
# CONSTRAINTS
# 
constraints =[
    cp.sum(h) == 1.0, # fully invested
    h >= 0.0 # long only
]

# The double quadratic optimization: Tracking Error Leash
tracking_error = cp.quad_form(h - h_b, cp.psd_wrap(V))
constraints.append(tracking_error <= max_tracking_error)


# CRA Tax Rules (The 30-Day Lockouts)
for i, permno in enumerate(permnos):
    if permno in do_not_buy:
        # FORWARD RULE: We harvested this recently. Cannot increase weight.
        constraints.append(h[i] <= h_current[i])  
        
    if permno in do_not_harvest:
        # BACKWARD RULE: We bought this recently. Cannot sell at a loss.
        constraints.append(h[i] >= h_current[i])


In [234]:
import warnings
# ==========================================
# SOLVING THE OPTIMIZER
# ==========================================

prob = cp.Problem(objective, constraints)
try:
    prob.solve()
    if h.value is None:
        raise ValueError("Solver failed to find a feasible solution.")
    
    # Clip floating point noise and strictly re-normalize to 1.0 
    # to avoid shorting from computational jitter 
    h_opt = np.clip(h.value, 0.0, 1.0)
    h_opt =  pd.Series(h_opt / np.sum(h_opt), index=permnos)
    
except Exception as e:
    warnings.warn(f"CVXPY Solver Error: {e}. Falling back to Benchmark weights.")
    h_opt = pd.Series(h_b, index=permnos)


In [235]:
raw_delta = (h_opt - h_current)
raw_delta

10104    0.000053
10107    0.000043
10138    0.000046
10145    0.000056
10147    0.000056
           ...   
92988    0.000047
93002    0.000050
93096    0.000046
93159   -0.000293
93422    0.000066
Length: 504, dtype: float64

In [236]:
# increased positions 
raw_delta[raw_delta > 0 ]

10104    0.000053
10107    0.000043
10138    0.000046
10145    0.000056
10147    0.000056
           ...   
92890    0.000045
92988    0.000047
93002    0.000050
93096    0.000046
93422    0.000066
Length: 479, dtype: float64

In [237]:
# positions increased more than 1e-4
raw_delta[(raw_delta > 0) & (raw_delta > 1e-4)]

12622    0.000120
88436    0.000119
dtype: float64

Tracking Error Variance relies on an L2 quadratic norm, which heavily penalizes concentrated deviations. Consequently, the solver prefers to **'smear'** the proxy reinvestment across the entire benchmark to diffuse the variance penalty.

While mathematically correct, this is not tradable in real-life.  Paying a bid-ask spread on a $5 trade destroys the tax alpha we just generated.

In [238]:
# positions harvested and the weight harvested 
raw_delta[(raw_delta < 0)]

11618   -0.000630
13936   -0.000452
15553   -0.000299
21776   -0.001471
21792   -0.000463
23026   -0.000750
24010   -0.000725
27828   -0.003287
39538   -0.000471
53613   -0.001656
59176   -0.004393
75100   -0.000596
78877   -0.000559
79089   -0.000225
81774   -0.001131
82298   -0.000216
82618   -0.000885
83435   -0.002070
84032   -0.001215
85072   -0.000433
89004   -0.000480
89641   -0.001001
90071   -0.000423
90162   -0.000193
93159   -0.000293
dtype: float64

In [239]:
# the portoflio positions in the harvested permnos 
h_opt[raw_delta[(raw_delta < 0)].index]

11618    0.000000e+00
13936    0.000000e+00
15553    0.000000e+00
21776    0.000000e+00
21792    0.000000e+00
23026    0.000000e+00
24010    0.000000e+00
27828    0.000000e+00
39538    0.000000e+00
53613    0.000000e+00
59176    0.000000e+00
75100    0.000000e+00
78877    0.000000e+00
79089    0.000000e+00
81774    1.397846e-11
82298    0.000000e+00
82618    0.000000e+00
83435    0.000000e+00
84032    0.000000e+00
85072    0.000000e+00
89004    0.000000e+00
89641    6.855108e-12
90071    0.000000e+00
90162    2.857980e-11
93159    0.000000e+00
dtype: float64

### Two issues are observed
#### 1. No partial harvesting:

I used the L2 tracking error in the constraints to induce partial harvesting. The optimizer fully harvested the positions in the harvesting candidates. No partial harvesting. We might be able to force partial harvesting if we reduce the max_tracking_error of 0.005 which  translates to a significant tracking error voaltility of sqrt(0.005) = 7.07%. A lower MET would leash the optimizer. 

#### 2. L2 smear: 
Executing 400 microscopic trades destroys tax alpha through bid-ask friction. Also, for we might be able to partially address the L2 smear problem by increaasing the L1 norm.


In [240]:
import itertools 
import time 
# ==========================================
# THE HYPERPARAMETER GRID SEARCH
# ==========================================
print("\n--- Running Convex Optimization Grid Search ---")

# Define the Grid
tev_budgets =[1e-6, 5e-5, 5e-4, 5e-3]       # From ultra-tight to loose tracking error
turnover_penalties =[0.001, 0.01, 0.1, 1.0] # From cheap trading to extremely expensive

results =[]

# Define standard constraints that don't change
base_constraints =[
    cp.sum(h) == 1.0,
    h >= 0.0
]
for i, permno in enumerate(permnos):
    if permno in do_not_buy:
        base_constraints.append(h[i] <= h_current[i])
    if permno in do_not_harvest:
        base_constraints.append(h[i] >= h_current[i])

# the quadratic constraint is fixed in every loop 
risk_form = cp.quad_form(h - h_b, cp.psd_wrap(V))

# Run the sweep
total_runs = len(tev_budgets) * len(turnover_penalties)
run_count = 1

for tev, lam in itertools.product(tev_budgets, turnover_penalties):
    print(f"Solving {run_count}/{total_runs} | TEV: {tev} | Penalty: {lam}...", end="\r")
    
    # Objective: Minimize Tax Penalty + (Lambda * Turnover)
    tax_penalty = h.T @ harvest_opportunity_cost
    turnover = cp.norm1(h - h_current)
    objective = cp.Minimize(tax_penalty + (lam * turnover))
    
    # Constraints: Base + Specific TEV Leash
    constraints = base_constraints + [risk_form <= tev]
    
    prob = cp.Problem(objective, constraints)
    
    try:
        # Let CVXPY auto-route the QCQP problem
        prob.solve()
        
        if h.value is None:
            raise ValueError("Infeasible")
            
        h_opt = np.clip(h.value, 0.0, 1.0)
        h_opt = h_opt / np.sum(h_opt)
        
        # Calculate Actionable Metrics
        raw_delta = h_opt - h_current
        
        # Filter out the 1e-9 computational dust
        actionable_sells = (raw_delta < -1e-4).sum()
        actionable_buys = (raw_delta > 1e-4).sum()
        
        # Did it partially or fully harvest?
        max_harvest_depth = raw_delta.min()  # The biggest single sell
        max_proxy_concentration = raw_delta.max() # The biggest single proxy buy
        
        results.append({
            'TEV_Budget': tev,
            'Turnover_Penalty': lam,
            'Actionable_Sells': actionable_sells,
            'Actionable_Buys': actionable_buys,
            'Max_Sell_Weight': max_harvest_depth,
            'Max_Buy_Weight': max_proxy_concentration,
            'Status': prob.status
        })
        
    except Exception as e:
        results.append({
            'TEV_Budget': tev, 'Turnover_Penalty': lam, 
            'Actionable_Sells': 0, 'Actionable_Buys': 0, 
            'Max_Sell_Weight': 0, 'Max_Buy_Weight': 0, 'Status': 'Failed/Frozen'
        })
        
    run_count += 1

# Convert to DataFrame for beautiful viewing
df_results = pd.DataFrame(results)

print("\n\n--- GRID SEARCH RESULTS ---")
display(df_results.sort_values(['TEV_Budget', 'Turnover_Penalty']))

# Let's create a quick pivot table to show Buys (Proxy Smear vs Concentration)
print("\n--- HEATMAP: Number of Proxy Buys (Smear vs. Sparsity) ---")
smear_matrix = df_results.pivot(index='TEV_Budget', columns='Turnover_Penalty', values='Actionable_Buys')
display(smear_matrix)


--- Running Convex Optimization Grid Search ---
Solving 16/16 | TEV: 0.005 | Penalty: 1.0.....

--- GRID SEARCH RESULTS ---


,TEV_Budget,Turnover_Penalty,Actionable_Sells,Actionable_Buys,Max_Sell_Weight,Max_Buy_Weight,Status
0,0.000001,0.001,25,2,-4.393470e-03,1.008924e-04,optimal
1,0.000001,0.010,25,2,-4.393470e-03,1.201143e-04,optimal
2,0.000001,0.100,1,0,-4.328196e-04,2.099593e-06,optimal
3,0.000001,1.000,0,0,3.192375e-09,4.116019e-07,optimal
4,0.000050,0.001,25,2,-4.393470e-03,1.009200e-04,optimal
5,0.000050,0.010,25,2,-4.393470e-03,1.196308e-04,optimal
6,0.000050,0.100,1,0,-4.328194e-04,2.098366e-06,optimal
7,0.000050,1.000,0,0,3.087744e-09,4.108717e-07,optimal
8,0.000500,0.001,25,2,-4.393470e-03,1.007649e-04,optimal
9,0.000500,0.010,25,2,-4.393470e-03,1.196102e-04,optimal



--- HEATMAP: Number of Proxy Buys (Smear vs. Sparsity) ---


Turnover_Penalty,0.001,0.010,0.100,1.000
TEV_Budget,,,,
0.000001,2,2,0,0
0.000050,2,2,0,0
0.000500,2,2,0,0
0.005000,2,2,0,0


In [241]:

# ==========================================
# THE HYPERPARAMETER GRID SEARCH
# ==========================================
print("\n--- Running Convex Optimization Grid Search ---")

# Define the Grid
tev_budgets =[1e-9, 5e-9, 1e-8, 1e-6]       # From ultra-tight to loose tracking error
turnover_penalties =[0.07, 0.8, 0.9, 0.1, 0.11, 0.12, 0.13] # From cheap trading to extremely expensive

results =[]

# Define standard constraints that don't change
base_constraints =[
    cp.sum(h) == 1.0,
    h >= 0.0
]
for i, permno in enumerate(permnos):
    if permno in do_not_buy:
        base_constraints.append(h[i] <= h_current[i])
    if permno in do_not_harvest:
        base_constraints.append(h[i] >= h_current[i])

# the quadratic constraint is fixed in every loop 
risk_form = cp.quad_form(h - h_b, cp.psd_wrap(V))

# Run the sweep
total_runs = len(tev_budgets) * len(turnover_penalties)
run_count = 1

for tev, lam in itertools.product(tev_budgets, turnover_penalties):
    print(f"Solving {run_count}/{total_runs} | TEV: {tev} | Penalty: {lam}...", end="\r")
    
    # Objective: Minimize Tax Penalty + (Lambda * Turnover)
    tax_penalty = h.T @ harvest_opportunity_cost
    turnover = cp.norm1(h - h_current)
    objective = cp.Minimize(tax_penalty + (lam * turnover))
    
    # Constraints: Base + Specific TEV Leash
    constraints = base_constraints + [risk_form <= tev]
    
    prob = cp.Problem(objective, constraints)
    
    try:
        # Let CVXPY auto-route the QCQP problem
        prob.solve()
        
        if h.value is None:
            raise ValueError("Infeasible")
            
        h_opt = np.clip(h.value, 0.0, 1.0)
        h_opt = h_opt / np.sum(h_opt)
        
        # Calculate Actionable Metrics
        raw_delta = h_opt - h_current
        
        # Filter out the 1e-9 computational dust
        actionable_sells = (raw_delta < -1e-4).sum()
        actionable_buys = (raw_delta > 1e-4).sum()
        
        # Did it partially or fully harvest?
        max_harvest_depth = raw_delta.min()  # The biggest single sell
        max_proxy_concentration = raw_delta.max() # The biggest single proxy buy
        
        results.append({
            'TEV_Budget': tev,
            'Turnover_Penalty': lam,
            'Actionable_Sells': actionable_sells,
            'Actionable_Buys': actionable_buys,
            'Max_Sell_Weight': max_harvest_depth,
            'Max_Buy_Weight': max_proxy_concentration,
            'Status': prob.status
        })
        
    except Exception as e:
        results.append({
            'TEV_Budget': tev, 'Turnover_Penalty': lam, 
            'Actionable_Sells': 0, 'Actionable_Buys': 0, 
            'Max_Sell_Weight': 0, 'Max_Buy_Weight': 0, 'Status': 'Failed/Frozen'
        })
        
    run_count += 1

# Convert to DataFrame for beautiful viewing
df_results = pd.DataFrame(results)

print("\n\n--- GRID SEARCH RESULTS ---")
display(df_results.sort_values(['TEV_Budget', 'Turnover_Penalty']))

# Let's create a quick pivot table to show Buys (Proxy Smear vs Concentration)
print("\n--- HEATMAP: Number of Proxy Buys (Smear vs. Sparsity) ---")
smear_matrix = df_results.pivot(index='TEV_Budget', columns='Turnover_Penalty', values='Actionable_Buys')
display(smear_matrix)


--- Running Convex Optimization Grid Search ---
Solving 28/28 | TEV: 1e-06 | Penalty: 0.13...

--- GRID SEARCH RESULTS ---


,TEV_Budget,Turnover_Penalty,Actionable_Sells,Actionable_Buys,Max_Sell_Weight,Max_Buy_Weight,Status
0,1.000000e-09,0.07,6,6,-2.239914e-03,1.620300e-03,optimal
3,1.000000e-09,0.10,7,6,-2.237703e-03,1.593421e-03,optimal
4,1.000000e-09,0.11,7,6,-2.238112e-03,1.595002e-03,optimal
5,1.000000e-09,0.12,7,6,-2.237538e-03,1.589523e-03,optimal
6,1.000000e-09,0.13,6,6,-2.237653e-03,1.588445e-03,optimal
1,1.000000e-09,0.80,5,5,-2.236978e-03,1.581417e-03,optimal
2,1.000000e-09,0.90,5,5,-2.236973e-03,1.581468e-03,optimal
7,5.000000e-09,0.07,5,4,-2.185193e-03,1.332650e-03,optimal
10,5.000000e-09,0.10,5,4,-2.191026e-03,1.281349e-03,optimal
11,5.000000e-09,0.11,5,4,-2.186966e-03,1.268150e-03,optimal



--- HEATMAP: Number of Proxy Buys (Smear vs. Sparsity) ---


Turnover_Penalty,0.07,0.10,0.11,0.12,0.13,0.80,0.90
TEV_Budget,,,,,,,
1.000000e-09,6,6,6,6,6,5,5
5.000000e-09,4,4,4,4,3,3,3
1.000000e-08,4,3,3,3,3,3,3
1.000000e-06,0,0,0,0,0,0,0


In [242]:
# setting the optimization hyperparams 

max_tracking_error= 5e-9

turnover_penalty= 0.10

# inititializing the optimization variable (portfolio weights)
h = cp.Variable(N)

# ==========================================
# BUILD THE OBJECTIVE AND CONSTRAINTS
# ==========================================        

# Opt. OBJECTIVE: Minimize (Holding Penalty + Tracking Error + Friction)

# Note: Since opportunity_cost_vector is positive for losers, 
# we MINIMIZE (h^T * opportunity_cost_vector).
# By minimizing a positive product, CVXPY forces h -> 0 for the losers.
tax_penalty = h.T @ harvest_opportunity_cost
turnover = cp.norm1(h - h_current)

objective = cp.Minimize(tax_penalty + (turnover_penalty * turnover))

#
# CONSTRAINTS
# 
constraints =[
    cp.sum(h) == 1.0, # fully invested
    h >= 0.0 # long only
]

# The double quadratic optimization: Tracking Error Leash
tracking_error = cp.quad_form(h - h_b, cp.psd_wrap(V))
constraints.append(tracking_error <= max_tracking_error)


# CRA Tax Rules (The 30-Day Lockouts)
for i, permno in enumerate(permnos):
    if permno in do_not_buy:
        # FORWARD RULE: We harvested this recently. Cannot increase weight.
        constraints.append(h[i] <= h_current[i])  
        
    if permno in do_not_harvest:
        # BACKWARD RULE: We bought this recently. Cannot sell at a loss.
        constraints.append(h[i] >= h_current[i])


In [243]:
# ==========================================
# SOLVING THE OPTIMIZER
# ==========================================

prob = cp.Problem(objective, constraints)
try:
    prob.solve()
    if h.value is None:
        raise ValueError("Solver failed to find a feasible solution.")
    
    # Clip floating point noise and strictly re-normalize to 1.0 
    # to avoid shorting from computational jitter 
    h_opt = np.clip(h.value, 0.0, 1.0)
    h_opt =  pd.Series(h_opt / np.sum(h_opt), index=permnos)
    
except Exception as e:
    warnings.warn(f"CVXPY Solver Error: {e}. Falling back to Benchmark weights.")
    h_opt = pd.Series(h_b, index=permnos)


In [244]:
raw_delta = (h_opt - h_current)
raw_delta

10104   -9.683777e-09
10107   -1.139499e-08
10138   -2.718862e-08
10145   -4.868783e-08
10147   -1.536587e-08
             ...     
92988    1.851193e-08
93002    2.495960e-08
93096   -2.953648e-09
93159   -2.944716e-05
93422    1.829249e-09
Length: 504, dtype: float64

In [245]:
# increased positions 
raw_delta[raw_delta > 0 ]

10299    6.282541e-10
10909    7.800655e-09
11552    2.439026e-10
11600    3.960897e-09
11762    1.146983e-08
             ...     
92293    7.986784e-09
92778    7.125702e-09
92988    1.851193e-08
93002    2.495960e-08
93422    1.829249e-09
Length: 128, dtype: float64

In [246]:
# Actionable BUYS
# positions increased more than 1e-4
raw_delta[(raw_delta > 0) & (raw_delta > 1e-4)]

12622    0.001281
60097    0.001271
69550    0.000137
88436    0.000686
dtype: float64

In [247]:
# positions harvested and the weight harvested 
raw_delta[(raw_delta < 0)]

10104   -9.683777e-09
10107   -1.139499e-08
10138   -2.718862e-08
10145   -4.868783e-08
10147   -1.536587e-08
             ...     
92655   -5.658256e-08
92709   -3.852412e-08
92890   -2.838281e-08
93096   -2.953648e-09
93159   -2.944716e-05
Length: 376, dtype: float64

In [248]:
# Actionable Sells
# positions harvested 
raw_delta[(raw_delta < 0) & (raw_delta < -1e-4)]

76149   -0.000121
79089   -0.000117
84032   -0.000119
85072   -0.000325
92156   -0.002191
dtype: float64

In [249]:
# Did we partially harvest? 
# the portoflio positions in the harvested permnos 
h_opt[raw_delta[(raw_delta < 0) & (raw_delta < -1e-4)].index]

76149    0.000275
79089    0.000108
84032    0.001096
85072    0.000108
92156    0.000065
dtype: float64

In [250]:
any(h_opt < 1e-6)

False

In [254]:
# ==========================================
# updating the portfolio positions based on opt. outputs h_opt
# translating Weights -> Target Shares
# ==========================================
print("\n--- Translating Weights to Target Shares ---")
MIN_TRADE_DOLLARS = 50.0  # Setting the minimum trade size (e.g. $50 CAD)


# initialize a dictionary for all the positions in the optimization output
target_shares = {}

for permno, weight in h_opt.items():
    if permno in prices_cad_40:
        price = prices_cad_40[permno]
        tgt_dollars = weight * total_aum
        
        # What do we currently own in dollars?
        cur_shares = positions.get(permno, {}).get('shares', 0.0)
        cur_dollars = cur_shares * price
        
        # Calculate the absolute dollar value of the proposed trade
        trade_dollar_value = abs(tgt_dollars - cur_dollars)
        
        # THE DUST FILTER: Only accept trades larger than $50
        if trade_dollar_value >= MIN_TRADE_DOLLARS:
            raw_shares = tgt_dollars / price
            target_shares[permno] = np.floor(raw_shares * 10000.0) / 10000.0
        else:
            # It's dust! Keep the target shares equal to current shares
            target_shares[permno] = cur_shares

print("Dust Filter Applied. Microscopic trades eliminated.")


--- Translating Weights to Target Shares ---
Dust Filter Applied. Microscopic trades eliminated.


In [255]:
realized_gains_cad = 0.0
realized_losses_cad = 0.0

# ==========================================
# Process SELLS
# ==========================================
print("\n--- Executing Sells (Funding Cash & Harvesting) ---")
harvested_count = 0

# We iterate over a list of keys to delete from the dict if shares hit 0
for permno in list(positions.keys()):
    tgt_shares = target_shares.get(permno, 0.0)
    cur_shares = positions[permno]['shares']
    delta = tgt_shares - cur_shares
    
    # If delta is negative, the optimizer told us to SELL
    if delta < -1e-6: # first dust filter 
        # safety check to not sell more than we own (floating point safety)
        shares_to_sell = min(abs(delta), cur_shares)
        
        acb = positions[permno]['acb_per_share']
        # if permno is dropped from the benchmark, just update with acb (will fix later)
        price = prices_cad_40.get(permno, acb)

        # 1. Update Cash Balance from cash generated from sells
        cash_cad += (shares_to_sell * price)
        
        # 2. Calculate Realized PnL for Tax Tracking
        realized_pnl = (price - acb) * shares_to_sell
        
        if realized_pnl < -1e-4:  # It's a loss!
            realized_losses_cad += abs(realized_pnl)
            
            #===================================
            # APPLY THE CRA FORWARD RULE
            #===================================
            # We harvested a loss, so we are locked out of buying this for 30 days
            superficial_loss_lockouts[permno] = day_40 + pd.Timedelta(days=30)
            harvested_count += 1
            print(f"HARVESTED: Permno {permno} | Sold {shares_to_sell:,.4f} shrs | Loss: -${abs(realized_pnl):,.2f}")
            
        else:  # It's a gain
            realized_gains_cad += realized_pnl
            print(f"SOLD (Gain): Permno {permno} | Sold {shares_to_sell:,.4f} shrs | Gain: +${realized_pnl:,.2f}")
            
        # 3. Deduct shares from our ledger
        positions[permno]['shares'] -= shares_to_sell
        
        # 4. Clean up empty positions if we sold the whole position
        if positions[permno]['shares'] < 1e-6:
            del positions[permno]

print(f"\n--- End of Sell Execution ---")
print(f"Successfully harvested {harvested_count} positions.")
print(f"Updated Cash Balance ready for Buys: ${cash_cad:,.2f}")


--- Executing Sells (Funding Cash & Harvesting) ---
HARVESTED: Permno 13936 | Sold 0.4130 shrs | Loss: -$6.76
HARVESTED: Permno 59176 | Sold 0.9733 shrs | Loss: -$6.90
SOLD (Gain): Permno 66800 | Sold 0.7621 shrs | Gain: +$2.60
SOLD (Gain): Permno 76149 | Sold 10.4336 shrs | Gain: +$0.00
HARVESTED: Permno 78877 | Sold 4.6235 shrs | Loss: -$13.87
HARVESTED: Permno 79089 | Sold 1.1939 shrs | Loss: -$27.36
HARVESTED: Permno 84032 | Sold 1.9131 shrs | Loss: -$17.24
HARVESTED: Permno 85072 | Sold 2.0868 shrs | Loss: -$94.85
SOLD (Gain): Permno 92156 | Sold 20.4954 shrs | Gain: +$0.00

--- End of Sell Execution ---
Successfully harvested 6 positions.
Updated Cash Balance ready for Buys: $3,866.67


In [256]:
# ==========================================
# Process BUYS
# ==========================================
print("\n--- Executing Buys (Proxy Reinvestment) ---")
bought_count = 0

for permno, tgt_shares in target_shares.items():
    # using double get to prevent crash if we are buying brand new stocks (e.g. in the benchmark but not in positions)
    cur_shares = positions.get(permno, {}).get('shares', 0.0)
    delta = tgt_shares - cur_shares
    
    # If delta is positive, the optimizer told us to BUY
    if delta > 1e-6:
        # Failsafe 1: Ensure we aren't legally locked out from buying (Forward Rule)
        if permno in superficial_loss_lockouts:
            print(f"WARNING: Optimizer tried to buy {permno} but it is locked out! Skipping.")
            continue
            
        price = prices_cad_40[permno]
        cost = delta * price
        
        # Failsafe 2: Ensure we have enough cash (Floating point/Margin protection)
        if cash_cad >= cost:
            cash_cad -= cost
        else:
            # If rounding dust caused a slight cash overdraft, buy exactly what we can afford
            affordable_shares = np.floor((cash_cad / price) * 10000.0) / 10000.0
            if affordable_shares <= 0: continue
            delta = affordable_shares
            cost = delta * price
            cash_cad -= cost

        # 1. CRA ACB Pooling Math
        if permno not in positions: # if we are initiating a position in a stock:
            positions[permno] = {'shares': delta, 'acb_per_share': price}
        else: 
            # if permno exists in the positions, we just update the number of shares and the acb
            old_shares = positions[permno]['shares']
            old_acb = positions[permno]['acb_per_share']
            new_shares = old_shares + delta
            
            # (Old Total Cost + New Total Cost) / New Total Shares
            new_acb = ((old_shares * old_acb) + cost) / new_shares
            positions[permno] = {'shares': new_shares, 'acb_per_share': new_acb}
            
        # Log the Backward Rule
        # If we buy this today, we cannot harvest a loss on it for the next 30 days.
        last_buy_dates[permno] = day_40
        bought_count += 1
        
        print(f"BOUGHT: Permno {permno} | Bought {delta:,.4f} shrs | Cost: ${cost:,.2f}")

print(f"\n--- End of Buy Execution ---")
print(f"Successfully reinvested cash into {bought_count} proxy positions.")
print(f"Final End of Day Cash Balance: ${cash_cad:,.2f}")


--- Executing Buys (Proxy Reinvestment) ---
BOUGHT: Permno 12622 | Bought 15.8050 shrs | Cost: $1,392.95
BOUGHT: Permno 60097 | Bought 14.2236 shrs | Cost: $1,381.79
BOUGHT: Permno 69550 | Bought 2.1571 shrs | Cost: $148.88
BOUGHT: Permno 88436 | Bought 6.9798 shrs | Cost: $746.28

--- End of Buy Execution ---
Successfully reinvested cash into 4 proxy positions.
Final End of Day Cash Balance: $196.77


In [192]:
# ==========================================
# Process BUYS
# ==========================================
print("\n--- Executing Buys (Proxy Reinvestment) ---")
bought_count = 0

for permno, tgt_shares in target_shares.items():
    # using double get to prevent crash if we are buying brand new stocks (e.g. in the benchmark but not in positions)
    cur_shares = positions.get(permno, {}).get('shares', 0.0)
    delta = tgt_shares - cur_shares
    
    # If delta is positive, the optimizer told us to BUY
    if delta > 1e-6:
        # Failsafe 1: Ensure we aren't legally locked out from buying (Forward Rule)
        if permno in superficial_loss_lockouts:
            print(f"WARNING: Optimizer tried to buy {permno} but it is locked out! Skipping.")
            continue
            
        price = prices_cad_40[permno]
        cost = delta * price
        
        # Failsafe 2: Ensure we have enough cash (Floating point/Margin protection)
        if cash_cad >= cost:
            cash_cad -= cost
        else:
            # If rounding dust caused a slight cash overdraft, buy exactly what we can afford
            affordable_shares = np.floor((cash_cad / price) * 10000.0) / 10000.0
            if affordable_shares <= 0: continue
            delta = affordable_shares
            cost = delta * price
            cash_cad -= cost

        # 1. CRA ACB Pooling Math
        if permno not in positions: # if we are initiating a position in a stock:
            positions[permno] = {'shares': delta, 'acb_per_share': price}
        else: 
            # if permno exists in the positions, we just update the number of shares and the acb
            old_shares = positions[permno]['shares']
            old_acb = positions[permno]['acb_per_share']
            new_shares = old_shares + delta
            
            # (Old Total Cost + New Total Cost) / New Total Shares
            new_acb = ((old_shares * old_acb) + cost) / new_shares
            positions[permno] = {'shares': new_shares, 'acb_per_share': new_acb}
            
        # Log the Backward Rule
        # If we buy this today, we cannot harvest a loss on it for the next 30 days.
        last_buy_dates[permno] = day_40
        bought_count += 1
        
        print(f"BOUGHT: Permno {permno} | Bought {delta:,.4f} shrs | Cost: ${cost:,.2f}")

print(f"\n--- End of Buy Execution ---")
print(f"Successfully reinvested cash into {bought_count} proxy positions.")
print(f"Final End of Day Cash Balance: ${cash_cad:,.2f}")


--- Executing Buys (Proxy Reinvestment) ---
BOUGHT: Permno 11762 | Bought 0.0001 shrs | Cost: $0.01
BOUGHT: Permno 12062 | Bought 0.0001 shrs | Cost: $0.02
BOUGHT: Permno 12622 | Bought 15.8050 shrs | Cost: $1,392.95
BOUGHT: Permno 13168 | Bought 0.0003 shrs | Cost: $0.03
BOUGHT: Permno 13407 | Bought 0.1044 shrs | Cost: $10.35
BOUGHT: Permno 13721 | Bought 0.0002 shrs | Cost: $0.01
BOUGHT: Permno 13788 | Bought 0.0002 shrs | Cost: $0.01
BOUGHT: Permno 14011 | Bought 0.0001 shrs | Cost: $0.01
BOUGHT: Permno 14297 | Bought 0.0001 shrs | Cost: $0.01
BOUGHT: Permno 19583 | Bought 0.0001 shrs | Cost: $0.01
BOUGHT: Permno 21207 | Bought 0.0007 shrs | Cost: $0.02
BOUGHT: Permno 23501 | Bought 0.0001 shrs | Cost: $0.00
BOUGHT: Permno 25785 | Bought 0.0020 shrs | Cost: $0.04
BOUGHT: Permno 27887 | Bought 0.0002 shrs | Cost: $0.02
BOUGHT: Permno 29102 | Bought 0.0007 shrs | Cost: $0.01
BOUGHT: Permno 38156 | Bought 0.0003 shrs | Cost: $0.02
BOUGHT: Permno 39917 | Bought 0.0002 shrs | Cost: $0.

In [ ]:

print("\n--- Translating Weights to Target Shares ---")
# initialize a dictionary for all the positions in the optimization output
target_shares = {}

for permno, weight in h_opt.items():
    # Only process meaningful weights and stocks we actually have prices for
    if weight > 1e-6 and permno in prices_cad_40:
        target_dollars = weight * total_aum
        raw_shares = target_dollars / prices_cad_40[permno]
        
        # Truncate to 4 decimals for Wealthsimple fractional shares
        # Flooring prevents microscopic rounding that could cause negative cash
        target_shares[permno] = np.floor(raw_shares * 10000.0) / 10000.0


--- Translating Weights to Target Shares ---


In [ ]:
realized_gains_cad = 0.0
realized_losses_cad = 0.0

# ==========================================
# Process SELLS
# ==========================================
print("\n--- Executing Sells (Funding Cash & Harvesting) ---")
harvested_count = 0

# We iterate over a list of keys to delete from the dict if shares hit 0
for permno in list(positions.keys()):
    tgt_shares = target_shares.get(permno, 0.0)
    cur_shares = positions[permno]['shares']
    delta = tgt_shares - cur_shares
    
    # If delta is negative, the optimizer told us to SELL
    if delta < -1e-6: # first dust filter 
        # safety check to not sell more than we own (floating point safety)
        shares_to_sell = min(abs(delta), cur_shares)
        
        acb = positions[permno]['acb_per_share']
        # if permno is dropped from the benchmark, just update with acb (will fix later)
        price = prices_cad_40.get(permno, acb)

        # 1. Update Cash Balance from cash generated from sells
        cash_cad += (shares_to_sell * price)
        
        # 2. Calculate Realized PnL for Tax Tracking
        realized_pnl = (price - acb) * shares_to_sell
        
        if realized_pnl < -1e-4:  # It's a loss!
            realized_losses_cad += abs(realized_pnl)
            
            #===================================
            # APPLY THE CRA FORWARD RULE
            #===================================
            # We harvested a loss, so we are locked out of buying this for 30 days
            superficial_loss_lockouts[permno] = day_40 + pd.Timedelta(days=30)
            harvested_count += 1
            print(f"HARVESTED: Permno {permno} | Sold {shares_to_sell:,.4f} shrs | Loss: -${abs(realized_pnl):,.2f}")
            
        else:  # It's a gain
            realized_gains_cad += realized_pnl
            print(f"SOLD (Gain): Permno {permno} | Sold {shares_to_sell:,.4f} shrs | Gain: +${realized_pnl:,.2f}")
            
        # 3. Deduct shares from our ledger
        positions[permno]['shares'] -= shares_to_sell
        
        # 4. Clean up empty positions if we sold the whole position
        if positions[permno]['shares'] < 1e-6:
            del positions[permno]

print(f"\n--- End of Sell Execution ---")
print(f"Successfully harvested {harvested_count} positions.")
print(f"Updated Cash Balance ready for Buys: ${cash_cad:,.2f}")


--- Executing Sells (Funding Cash & Harvesting) ---
SOLD (Gain): Permno 10104 | Sold 0.0002 shrs | Gain: +$0.00
HARVESTED: Permno 10107 | Sold 0.0003 shrs | Loss: -$0.00
SOLD (Gain): Permno 10138 | Sold 0.0003 shrs | Gain: +$0.00
SOLD (Gain): Permno 10145 | Sold 0.0005 shrs | Gain: +$0.01
SOLD (Gain): Permno 10147 | Sold 0.0005 shrs | Gain: +$0.00
HARVESTED: Permno 10516 | Sold 0.0006 shrs | Loss: -$0.00
SOLD (Gain): Permno 10696 | Sold 0.0002 shrs | Gain: +$0.00
SOLD (Gain): Permno 11308 | Sold 0.0006 shrs | Gain: +$0.00
SOLD (Gain): Permno 11404 | Sold 0.0002 shrs | Gain: +$-0.00
HARVESTED: Permno 11618 | Sold 0.3901 shrs | Loss: -$1.72
HARVESTED: Permno 11674 | Sold 0.0002 shrs | Loss: -$0.00
SOLD (Gain): Permno 11703 | Sold 0.0003 shrs | Gain: +$0.00
SOLD (Gain): Permno 11850 | Sold 0.0013 shrs | Gain: +$0.00
SOLD (Gain): Permno 11955 | Sold 0.0009 shrs | Gain: +$0.01
SOLD (Gain): Permno 12052 | Sold 0.0003 shrs | Gain: +$0.00
SOLD (Gain): Permno 12073 | Sold 0.0030 shrs | Gain: +

### The Incidental Wash Sale Trap.
1. Why 56 Harvests instead of 25?
We correctly identified 25 candidates that crossed the strict -5% threshold. We assigned those 25 a heavy Tax Penalty in the objective function, so the optimizer (partially) sold them as instructed.
Where did the other 31 come from?
**The "L2 Proxy Smear"** The optimizer is making microscopic trades (e.g., selling 0.0003 shares) across the portfolio to re-align Tracking Error.
**We have incidental harvest!** If the optimizer decides to shave 0.0003 shares off a stock that happens to be down -2% (which didn't trigger the scanner), the Ledger records that the price is lower than the ACB.
Ledger logs a capital loss of $0.00 and locks that stock out for 30 days.

**This is not my desired output** We triggered a 30-day CRA forward lockout on 31 stocks for literally zero pennies of tax benefit. If one of those 31 stocks crashes by 20% tomorrow, we are legally blocked from harvesting it! NOT GOOD!

2. The Solution: **The Minimum Trade Filter (Squashing the Dust)**
We need to tell the Execution Layer: "If the dollar value of the trade is less than $50, do not execute it. It is just optimization dust."
